<a href="https://colab.research.google.com/github/haider6tahir/Machine-Learning-Journey/blob/main/ann.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch

# Input
x = torch.tensor([
    [1.0],
    [0.5],
    [-1.5]
])

# Weights
W1 = torch.tensor([
    [0.2,  0.4, -0.5],
    [-0.3, 0.1,  0.2],
    [0.5, -0.2,  0.1],
    [0.7,  0.3, -0.1]
])

# Bias
b1 = torch.tensor([
    [0.1],
    [0.2],
    [-0.1],
    [0.3]
])

# Hidden -> Output
W2 = torch.tensor([
    [0.6, -0.4, 0.8, 0.2]
])

b2 = torch.tensor([
    [0.1]
])

# Forward pass
Z1 = W1 @ x + b1
A1 = torch.sigmoid(Z1)

Z2 = W2 @ A1 + b2
A2 = torch.sigmoid(Z2)

print("Z1 =", Z1)
print("A1 =", A1)
print("Z2 =", Z2)
print("A2 =", A2)
print("Shape of Z1 = ", Z1.shape)
print("Shape of A1 = ", A1.shape)
print("Shape of Z2 = ", Z2.shape)
print("Shape of A2 = ", A2.shape)

Z1 = tensor([[ 1.2500],
        [-0.3500],
        [ 0.1500],
        [ 1.3000]])
A1 = tensor([[0.7773],
        [0.4134],
        [0.5374],
        [0.7858]])
Z2 = tensor([[0.9881]])
A2 = tensor([[0.7287]])
Shape of Z1 =  torch.Size([4, 1])
Shape of A1 =  torch.Size([4, 1])
Shape of Z2 =  torch.Size([1, 1])
Shape of A2 =  torch.Size([1, 1])


In [3]:
# One variable Regression
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

In [4]:
SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

PyTorch: 2.11.0+cpu
Device: cpu


In [5]:
df1 = fetch_california_housing(as_frame=True).frame

print(df1.shape)
df1.head()

(20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [6]:
feature_cols = [c for c in df1.columns if c != "MedHouseVal"]

X = df1[feature_cols].values.astype(np.float32)

y = df1[["MedHouseVal"]].values.astype(np.float32)

print(X.shape)
print(y.shape)

(20640, 8)
(20640, 1)


In [7]:
# 4. Split the data
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=SEED
)

X_va, X_te, y_va, y_te = train_test_split(
    X_tmp, y_tmp,
    test_size=0.50,
    random_state=SEED
)

print(X_tr.shape)
print(X_va.shape)
print(X_te.shape)

(14448, 8)
(3096, 8)
(3096, 8)


In [8]:
# Scale the inputs
scaler = StandardScaler().fit(X_tr)

X_tr = scaler.transform(X_tr).astype(np.float32)
X_va = scaler.transform(X_va).astype(np.float32)
X_te = scaler.transform(X_te).astype(np.float32)

In [10]:
# Here we Train the actual neural network.
# nn.linear means each 8 inputs and 64 neurons (internally w = 64 * 8 and bias of 64 neurons are 64 ),
#  then 64 inputs and 32 neurons and then 32 neurons and 1 output that is a number
model1 = nn.Sequential(
    nn.Linear(8, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1)
).to(device)

In [13]:
# Loss function
loss_fn = nn.MSELoss()

In [14]:
# Optimizer
# This is where learning happens.
# Adam will modify the network's:
# weights
# +
# biases

# so that the loss becomes smaller.

optimizer = torch.optim.Adam(
    model1.parameters(),
    lr=1e-3
)

In [16]:
# Create batches from our training data
def make_loader(X, y, batch_size=256, shuffle=False):
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(X_tr, y_tr, shuffle=True)
val_loader   = make_loader(X_va, y_va)
test_loader  = make_loader(X_te, y_te)

In [17]:
xb, yb = next(iter(train_loader))

print("xb shape:", xb.shape)
print("yb shape:", yb.shape)

xb shape: torch.Size([256, 8])
yb shape: torch.Size([256, 1])


In [18]:
xb = xb.to(device)
yb = yb.to(device)

pred = model1(xb)

print("prediction shape:", pred.shape)

prediction shape: torch.Size([256, 1])


In [21]:
EPOCHS = 30

for epoch in range(EPOCHS):

    model1.train()

    for xb, yb in train_loader:

        xb = xb.to(device)
        yb = yb.to(device)

        # 1. Forward
        pred = model1(xb)

        # 2. Loss
        loss = loss_fn(pred, yb)

        # 3. Clear old gradients
        optimizer.zero_grad()

        # 4. Backpropagation
        loss.backward()

        # 5. Update weights
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss = {loss.item():.4f}")

Epoch 1, Loss = 1.1316
Epoch 2, Loss = 0.6918
Epoch 3, Loss = 0.8511
Epoch 4, Loss = 0.5229
Epoch 5, Loss = 0.4971
Epoch 6, Loss = 0.4521
Epoch 7, Loss = 0.4687
Epoch 8, Loss = 0.4907
Epoch 9, Loss = 0.5703
Epoch 10, Loss = 0.3998
Epoch 11, Loss = 0.4119
Epoch 12, Loss = 0.3885
Epoch 13, Loss = 0.4088
Epoch 14, Loss = 0.4814
Epoch 15, Loss = 0.2677
Epoch 16, Loss = 0.3999
Epoch 17, Loss = 0.3049
Epoch 18, Loss = 0.3908
Epoch 19, Loss = 0.3783
Epoch 20, Loss = 0.4314
Epoch 21, Loss = 0.4018
Epoch 22, Loss = 0.5710
Epoch 23, Loss = 0.4056
Epoch 24, Loss = 0.3395
Epoch 25, Loss = 0.2824
Epoch 26, Loss = 0.3230
Epoch 27, Loss = 0.3261
Epoch 28, Loss = 0.2436
Epoch 29, Loss = 0.3779
Epoch 30, Loss = 0.2537


In [22]:
print(model1[0].weight.grad.shape)
print(model1[2].weight.grad.shape)
print(model1[4].weight.grad.shape)

torch.Size([64, 8])
torch.Size([32, 64])
torch.Size([1, 32])


In [23]:
# Now evaluate the model

model1.eval()

with torch.no_grad():
    test_pred = model1(torch.tensor(X_te).to(device))

test_pred = test_pred.cpu().numpy()

print("Prediction shape:", test_pred.shape)
print("Actual shape:", y_te.shape)

Prediction shape: (3096, 1)
Actual shape: (3096, 1)


In [24]:
# Calculate regression metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_te, test_pred)
rmse = np.sqrt(mean_squared_error(y_te, test_pred))
r2 = r2_score(y_te, test_pred)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

MAE : 0.3982263207435608
RMSE: 0.5684155832923565
R²  : 0.7556577920913696
